# Lekcia 13 - Pamäť agenta


## Nastavenie

Tento poznámkový blok demonštruje, ako vytvoriť cestovného rezervačného agenta s **perzistentnou pamäťou** pomocou **Microsoft Agent Framework** (MAF).

Naučíte sa, ako rôzne typy agentovej pamäte — pracujúca, krátkodobá a dlhodobá — ovplyvňujú to, ako si agent uchováva a používa informácie počas viacerých konverzácií.

**Predpoklady:**
- Projekt Azure AI Foundry s nasadeným chatovacím modelom (napr. `gpt-4o-mini`).
- Prihlásený v Azure CLI — spustite v termináli `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — koncový bod vášho projektu Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — názov vášho nasadeného modelu.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy pamäti agenta

AI agenti môžu využívať rôzne druhy pamäte, z ktorých každá slúži inému účelu:

### Pracovná pamäť
Samotná konverzačná vlákna — správy vymenené v jednej relácii. Agent sa môže odvolať na predchádzajúce správy v tom istom vlákne, aby udržal koherentnosť. V MAF sa toto vytvára pomocou **`agent.create_session()`**, ktorá vracia `AgentSession`.

### Krátkodobá pamäť
Informácie, ktoré pretrvávajú počas trvania úlohy alebo relácie, ale nie sú trvalo uložené. Napríklad agent môže zhromažďovať fakty počas viackolovej plánovacej konverzácie a použiť ich na vytvorenie konečného itinerára.

### Dlhodobá pamäť
Preferencie a fakty, ktoré pretrvávajú **naprieč reláciami**. Vracajúci sa používateľ by nemal opakovane uvádzať svoje diétny obmedzenia alebo cestovný štýl. Dlhodobá pamäť je zvyčajne podporovaná externým úložiskom — databázou, súborom alebo vektorovým indexom — a sprístupnená agentovi cez nástroje.


## Pracovná pamäť so sessionmi

Najjednoduchšou formou pamäti je konverzačná session. Keď odovzdáte tú istú session (vytvorenú cez `agent.create_session()`) do nasledujúcich volaní `agent.run()`, agent vidí celú históriu tejto konverzácie a môže si vybaviť skoršie detaily.

Vytvorme si cestovného agenta a demonštrujme pracovnú pamäť.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent si správne vybavil rozpočet, pretože obe správy zdieľajú tú istú reláciu. Toto je **pracovná pamäť** — existuje iba počas životnosti relácie.

### Čo sa stane s novým vláknom?

Ak vytvoríme **novú** reláciu, agent si nepamätá predchádzajúci rozhovor:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzor dlhodobej pamäti

Na zapamätanie preferencií používateľa **naprieč reláciami** potrebujeme trvalé úložisko, ktoré žije mimo vlákna konverzácie. Agent pristupuje k tomuto úložisku cez **nástroje** — funkcie, ktoré môže volať na uloženie a získanie informácií.

Nižšie implementujeme jednoduché pamäťové úložisko preferencií (v produkcii by ste ho podporili databázou alebo vektorovým indexom) a sprístupníme ho ako nástroje, ktoré agent môže používať.

### Architektúra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenár 1 — Prvýkrát používateľ rezervuje výročnú cestu

Sarah navštívi stránku prvýkrát. Agent by mal uložiť jej preferencie cez nástroje a použiť ich na odporúčanie hotelov.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenár 2 — Sarah sa vracia o niekoľko týždňov neskôr

Sarah začína **úplne novú tému** (simulujúc novú reláciu). Pracovná pamäť je prázdna, ale dlhodobé úložisko preferencií má stále jej informácie. Agent by ich mal vyhľadať a použiť na personalizáciu odporúčaní.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zhrnutie

V tejto lekcii ste sa naučili tri typy pamäte agenta a ako ich implementovať pomocou Microsoft Agent Framework:

| Typ pamäte | Mechanizmus MAF | Životnosť |
|---|---|---|
| **Pracovná** | `agent.create_session()` | Jedna konverzácia |
| **Krátkodobá** | Nahratá kontextová história v rámci vlákna | Jeden úloh / relácia |
| **Dlhodobá** | Externý úložný priestor prístupný cez funkcie `@tool` | Medzi reláciami |

### Hlavné body
1. **`agent.create_session()`** poskytuje pracovnú pamäť — agent vidí celú históriu konverzácie v rámci jednej relácie.
2. **Nové relácie strácajú kontext** — bez dlhodobej pamäte si agent nepamätá minulé konverzácie.
3. **Funkcie `@tool`** preklenujú tento rozdiel — umožňujú agentovi ukladať a získavať informácie z trvalého úložiska.
4. **Personalizácia sa zlepšuje v čase** — čím viac sa uloží preferencií, tým lepšie sú odporúčania agenta.

### Skutočné využitia
- **Zákaznícky servis**: Pamätať si históriu a preferencie zákazníka
- **Osobní asistenti**: Udržiavať kontext počas dní alebo týždňov
- **Zdravotníctvo**: Sledovať informácie o pacientovi a preferencie
- **E-commerce**: Personalizované nakupovanie na základe histórie

### Ďalšie kroky
- Nahradiť pamäť v dict databázou alebo vektorovým úložiskom (napr. Azure AI Search)
- Pridať vypršanie platnosti pamäte pre časovo citlivé informácie
- Vytvoriť systémy s viacerými agentmi s zdieľanou pamäťou
- Preskúmať poznámkový blok Cognee pre pamäť založenú na znalostnom grafe


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, prosím, majte na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho rodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
